## IMPORT & SETUP


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os
from google.colab import drive, files

print(os.getcwd())
drive.mount("/content/drive")
df = pd.read_csv(
    "/content/drive/MyDrive/SE-dataset/cleaned_diseases_symptoms.csv"
)
print(f"Shape: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")

/content
Mounted at /content/drive
Shape: (101485, 377)
Unique diseases: 658


Import required libraries, pandas, numpy, scikit-learn modules (RandomForestClassifier, train_test_split, LabelEncoder, evaluation metrics), joblib, and os; then mounts Google Drive and loads the cleaned dataset (cleaned_diseases_symptoms.csv). Output confirms the data loaded with the shape (101723, 377) and 773 unique diseases.

## SEPARATING FEATURES AND TARGET


In [2]:
# Filter out diseases with fewer than 20 samples (minimum for stratified split)
MIN_SAMPLES = 20

counts = df['diseases'].value_counts()
df = df[df['diseases'].isin(counts[counts >= MIN_SAMPLES].index)].copy()

print(f"Shape after filter: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")

X = df.drop(columns=["diseases"])
y = df["diseases"]

Shape after filter: (99926, 377)
Unique diseases: 512


 This applies a second filter to drop any disease with fewer than 20 samples, since stratified train/val/test splitting requires a minimum number of samples per class. After filtering, the dataset shrinks to (101485, 377) with 658 unique diseases. The data is then split into features(X, for all symptom columns) and target (y, the diseases column) in preparation for model training

## ENCODING TARGET LABELS


In [3]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
os.makedirs('../models', exist_ok = True)
joblib.dump(le, '../models/label_encoder.pk')

['../models/label_encoder.pk']

This section converts the disease names into numeric values using LabelEncoder, since scikit-learn models require numeric targets. The fitted encoder is saved to models/label_encoder.pk so the same encoding can be reused later to decode predictions back into disease names.

## Train / val / test split (60/20/20 stratified)


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (59955, 376) | Val: (19985, 376) | Test: (19986, 376)


This section splits the dataset into training (60%), validation (20%), and test (20%) sets using train_test_split, applied twice; first to carve out the training set, then to split the remainder evenly into validation and test. The stratify parameter ensures each split preserves the same disease-class proportions as the full dataset. Resulting shapes: (59955, 376), Val: (19985, 376), Test: (19986, 376), each with 376 feature columns.

## TRAIN RANDOM FOREST


The original version of the Random Forest model used 100 trees and an unconstrained first run (no `max_depth`) as well using a min of 2 samples to split a node. The results had a good Train accuracy of 90.7%, with Val and Test hovering around 82%

The tuned version used more trees, 200 of them, as well as reducing the `max_depth` to 20 to reduce overfitting and having `min_samples_leaf` as 5. The val/test gap did close but at the cost of 10% accuracy (76% train acc vs 73% val/test acc.)

Decided on a middle ground where the classifier used the same 200 `n_estimators` and unconstrained the depth and changing the `min_samples_leaf` to 3, slightly more than default.

In [9]:
rf = RandomForestClassifier(        # Main hyperparameters worth tuning
    n_estimators=200,               # more trees
    max_depth=None,                 # unconstrained depth
    min_samples_leaf=3,             # min samples at lead node
    min_samples_split = 5,
    max_features = 'sqrt',
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,                       # parallelism
)

print("Training...")
rf.fit(X_train, y_train)
print("Done")

Training...
Done


A RandomForestClassifier is configured with regularization settings to reduce overfitting: 200 trees, unconstrained depth, but min_samples_leaf = 3 and min_samples_split = 5, max_features = 'sqrt' which means that each tree only considers random subset of features, and class_weight = "balanced, which compensates for any remaining class imbalances. The model trains on X_train/y_train and completes successfully.

## EVALUATE

In [10]:
for name, X_set, y_set in [("Train", X_train, y_train),
                           ("Val", X_val, y_val),
                           ("Test", X_test, y_test)]:
  pred = rf.predict(X_set)
  acc = accuracy_score(y_set, pred)
  f1 = f1_score(y_set, pred, average = 'macro', zero_division = 0)
  print(f"{name} - Accuracy: {acc} | Macro-F1: {f1}")

Train - Accuracy: 0.891652072387624 | Macro-F1: 0.8692462242916863
Val - Accuracy: 0.8401801351013259 | Macro-F1: 0.8129184707840182
Test - Accuracy: 0.836935855098569 | Macro-F1: 0.8083731847376174


The model is evaluated on Train, Validation, and Test sets using accuracy and macro-F1. Compared to a less-regularized version, train accuracy dropped while the train/validation gap narrowed from 8% to 5%. This means the added regularization successfully reduced overfitting. The model generalizes a bit better to unseen data, though a small train/val gap still remains, which is acceptable but suggests there is still modest room to reduce overitting further if needed.

Upon regularisation, we have seen improvements in:
* Train accuracy dropped from 90.7% to 89.1%, meaning there's less memorisation and generalisation remain unchanged
* The train/val gap narrowed from approximately 8% to 5%, a bit of overfitting but it's okay

This will be our final model

## SAVING MODEL

In [11]:
joblib.dump(rf, '/content/drive/MyDrive/SE-dataset/random_forest.pkl')
joblib.dump(le, '/content/drive/MyDrive/SE-dataset/label_encoder.pkl')
print("Model saved")

Model saved


With the Random Forest confirmed as the final model, both it and the label encoder are  saved to Google Drive using joblib.dump(), random_forest.pkl and label_encoder.pkl. This allows the trained model to be reloaded later for predictions without needing to retrain it.